# Cuaderno 4: ¿cuánto se acopla la caída con el estado basal? (VitalDB)

**Autor:** Aly Gallo · Continúa los cuadernos 1, 2 y 3.

**Pregunta:** en el cuaderno 3 todo dependía del *acoplamiento* (c): si los pacientes con peor estado basal caen más fuerte con la operación. Con c = 0 el método M2 casi funciona; con c = 0,3 nada funciona. Aquí se **mide c con datos reales**, usando solo la profundidad de la caída del BSI tras operar (no se usa ningún desenlace de salud ni recuperación).

**Cómo:** el modelo del simulador dice que la profundidad es D = D0·(1 + c·z), con z = BSI basal verdadero. El nadir (punto más bajo) es entonces ≈ z − D, y su pendiente sobre el basal es 1 − D0·c. Se estima esa pendiente, se divide por la profundidad media D0 y se corrige por el ruido de la línea base (atenuación). Es una **estimación aproximada**: da un rango, no un número exacto.

**Luego:** con el c medido se recalcula el poder de M0 y M2 con n = 150 (lo que hay con 2 extracciones previas), n = 738, y n = 5 000 y 10 000 (escala INSPIRE, solo con el c estimado), para ver cuántos pacientes harían falta.

In [ ]:
import sys, datetime, json, time, warnings
import numpy as np, pandas as pd
from statsmodels.regression.quantile_regression import QuantReg
warnings.simplefilter("ignore")
print("Fecha:", datetime.date.today(), "| Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

W = 7.0; PRE_DIAS = 7; MIN_POST = 3
SEMILLA = 2026
N_REPS = 100                  # repeticiones por escenario
TECHOS = [0.5, 1.0]           # techo moderado y fuerte
ACOPLAMIENTOS = [0.0, 0.3]
D0, T0, TAU = 1.0, 2.0, 2.0
UMBRAL_CAIDA = 0.5
pruebas = ["crp", "alb", "wbc"]


## Paso 1. Datos y cohorte real (igual que en el cuaderno 2)

In [ ]:
lab = pd.read_csv("https://api.vitaldb.net/labs")
cases = pd.read_csv("https://api.vitaldb.net/cases")
d = lab.merge(cases[["caseid", "subjectid", "opend"]], on="caseid", how="left")
d["dias_post"] = (d.dt - d.opend) / 86400
d["dias_pre"] = -d.dt / 86400

conjuntos = []
for t in pruebas:
    x = d[d.name == t]
    pre = set(x[(x.dt <= 0) & (x.dias_pre <= PRE_DIAS)].caseid)
    p = x[(x.dias_post >= 0) & (x.dias_post <= W)].groupby("caseid").dias_post.agg(["size", "max"])
    post = set(p[(p["size"] >= MIN_POST) & (p["max"] >= W / 2)].index)
    conjuntos.append(pre & post)
coh = set.intersection(*conjuntos)
c = cases[cases.caseid.isin(coh)].sort_values("caseid").drop_duplicates("subjectid", keep="first")
ids = c.caseid.tolist()
print("Cirugias en la cohorte:", len(coh), "| pacientes unicos:", len(ids), "(esperado: 774 y 738)")

base = {}
for t in pruebas:
    x = d[(d.name == t) & d.caseid.isin(ids) & (d.dt <= 0) & (d.dias_pre <= PRE_DIAS)]
    base[t] = x.sort_values("dt").groupby("caseid").result.last()
B = pd.DataFrame(base).loc[ids]
# parametros de estandarizacion (se reutilizan en el Paso A)
def comp(df):
    return pd.concat([-np.log(df.crp + 0.1), df.alb, -np.log(df.wbc)], axis=1)
Z = comp(B)
mu_z, sd_z = Z.mean(), Z.std()
raw = ((Z - mu_z) / sd_z).mean(axis=1)
mu_raw, sd_raw = raw.mean(), raw.std()
bsi_real = ((raw - mu_raw) / sd_raw).values

xc = d[(d.name == "crp") & d.caseid.isin(ids) & (d.dias_post >= 0) & (d.dias_post <= W)]
tiempos = [np.sort(xc[xc.caseid == i].dias_post.values) for i in ids]
m = max(len(t) for t in tiempos)
T = np.full((len(ids), m), np.nan)
for k, t in enumerate(tiempos):
    T[k, :len(t)] = t
N = len(ids)


## Paso A. Estimar el acoplamiento c
Por cada prueba se toma el nadir (promedio de los 2 valores más bajos del componente transformado, dentro de los 7 días posteriores a la cirugía). El nadir del BSI es el promedio de los 3 nadires estandarizados. Se compara con el BSI basal. Es aproximado porque los 3 nadires pueden ocurrir en días distintos (tiende a exagerar la profundidad).

In [ ]:
nad = {}
for t in pruebas:
    x = d[(d.name == t) & d.caseid.isin(ids) & (d.dias_post >= 0) & (d.dias_post <= W)].copy()
    v = x.result
    if t == "crp": v = -np.log(v + 0.1)
    if t == "wbc": v = -np.log(v)
    x["v"] = (v - mu_z[t]) / sd_z[t]
    nad[t] = x.groupby("caseid").v.apply(lambda s: np.sort(s.values)[:2].mean())
Nd = pd.DataFrame(nad).loc[ids]
raw_n = Nd.mean(axis=1)
nadir_bsi = ((raw_n - mu_raw) / sd_raw).values
base_obs = bsi_real
ok = ~np.isnan(nadir_bsi)
print("Pacientes con nadir calculable:", int(ok.sum()))
prof = base_obs[ok] - nadir_bsi[ok]
print("Profundidad media D0 (unidades de SD del BSI):", round(prof.mean(), 2), "| SD:", round(prof.std(), 2))

def pendiente(b, n_):
    X = np.column_stack([np.ones(len(b)), b])
    return np.linalg.lstsq(X, n_, rcond=None)[0][1]

b_n = pendiente(base_obs[ok], nadir_bsi[ok])
print("Pendiente nadir ~ basal (cruda):", round(b_n, 3))
def c_est(b_n, D0_, sigma):
    lam = 1 / (1 + sigma ** 2)          # confiabilidad de la linea base
    return (1 - b_n / lam) / D0_
res = {}
for sg in [0.0, 0.15, 0.32]:
    res[sg] = c_est(b_n, prof.mean(), sg)
print("Acoplamiento estimado c segun el ruido de la linea base:", {k: round(v, 2) for k, v in res.items()})

rng_b = np.random.default_rng(SEMILLA)
boot = {sg: [] for sg in res}
bo, nb_ = base_obs[ok], nadir_bsi[ok]
for _ in range(1000):
    i = rng_b.integers(0, len(bo), len(bo))
    bb = pendiente(bo[i], nb_[i]); dd = (bo[i] - nb_[i]).mean()
    for sg in res:
        boot[sg].append(c_est(bb, dd, sg))
print("\nIntervalo 95% (bootstrap de pacientes):")
for sg in res:
    lo, hi = np.percentile(boot[sg], [2.5, 97.5])
    print(f"  ruido linea base {sg}: c = {res[sg]:.2f}  [{lo:.2f}, {hi:.2f}]")
C_HAT = float(np.clip(res[0.32], 0.0, 1.0)); C_BAJO = float(np.clip(np.percentile(boot[0.32], 2.5), 0, 1)); C_ALTO = float(np.clip(np.percentile(boot[0.32], 97.5), 0, 1))
print("\nValores de c que se usan en el Paso B:", round(C_BAJO, 2), round(C_HAT, 2), round(C_ALTO, 2))


## Paso B. Simulador con los tres métodos
Cada paciente simulado tiene dos mediciones previas independientes (`base1` y `base2`) del mismo estado verdadero:
- **M0:** predictor `base1`; resultado `(final − nadir)/(base1 − nadir)`.
- **M1:** predictor `base1`; resultado `final` (nivel final, sin cociente).
- **M2:** predictor `base1`; resultado `(final − nadir)/(base2 − nadir)`.
En todos, la prueba es Δβ = β(0,90) − β(0,50) de una regresión cuantílica de resultado sobre el predictor. Se analizan solo pacientes con caída observada > 0,5 SD (mismo criterio para M0 y M1 con `base1`; M2 con `base2`).

In [ ]:
def generar(rng, idx, ruido, acople, techo):
    n = len(idx); Tn = T[idx]; mask = ~np.isnan(Tn)
    Bt = bsi_real[idx]
    D = np.clip(D0 * (1 + acople * Bt), 0.3, None)
    u = rng.beta(4, 2, n)
    r = u if techo is None else np.minimum(u, np.clip(0.95 + techo * Bt, 0.05, 1.0))
    k = 1 - np.exp(-(W - T0) / TAU)
    t = np.where(mask, Tn, 0.0)
    rec = r[:, None] * (1 - np.exp(-(t - T0) / TAU)) / k
    q = np.where(t <= T0, t / T0, 1 - rec)
    x = Bt[:, None] - D[:, None] * q
    obs = np.where(mask, x + rng.normal(0, ruido, x.shape), np.nan)
    base1 = Bt + rng.normal(0, ruido, n)
    base2 = Bt + rng.normal(0, ruido, n)
    srt = np.sort(np.where(mask, obs, np.inf), axis=1)
    nadir = srt[:, :2].mean(axis=1)
    last = np.nanargmax(np.where(mask, Tn, -np.inf), axis=1)
    fin = obs[np.arange(n), last]
    return base1, base2, nadir, fin

def resultado(metodo, base1, base2, nadir, fin):
    if metodo == "M0":
        ok = (base1 - nadir) > UMBRAL_CAIDA
        R = np.clip((fin - nadir) / np.where(ok, base1 - nadir, 1), -0.5, 1.5)
    elif metodo == "M1":
        ok = (base1 - nadir) > UMBRAL_CAIDA
        R = fin
    else:  # M2
        ok = (base2 - nadir) > UMBRAL_CAIDA
        R = np.clip((fin - nadir) / np.where(ok, base2 - nadir, 1), -0.5, 1.5)
    return base1[ok], R[ok]

def delta_beta(S, R):
    X = np.column_stack([np.ones(len(S)), S])
    b90 = QuantReg(R, X).fit(q=0.90, max_iter=1000).params[1]
    b50 = QuantReg(R, X).fit(q=0.50, max_iter=1000).params[1]
    return b90 - b50


## Paso B. Poder de M0 y M2 con el acoplamiento medido
Se corre el simulador con c = límite inferior, estimado y límite superior; ruido 0,15 y 0,32; y distintos tamaños de muestra (n = 150, 738, 5 000, 10 000; los dos últimos solo con el c estimado). Umbral crítico = percentil 95 del mundo sin techo, igual que antes. Tarda unos 12 a 15 minutos.

In [ ]:
rng = np.random.default_rng(SEMILLA + 4)
CS = sorted(set([round(C_BAJO, 2), round(C_HAT, 2), round(C_ALTO, 2)]))
RUIDOS = [0.15, 0.32]
TAMS = [150, 738, 5000, 10000]
filas = []; t_ini = time.time()
def reps_for(n):
    return N_REPS if n <= 738 else (30 if n <= 5000 else 15)
for n_ in TAMS:
    R_ = reps_for(n_)
    for ruido in RUIDOS:
        for cc in (CS if n_ <= 738 else [round(C_HAT, 2)]):
            for met in ["M0", "M2"]:
                nulo = []
                for _ in range(R_):
                    idx = rng.choice(N, n_, replace=True)
                    S, R = resultado(met, *generar(rng, idx, ruido, cc, None)); nulo.append(delta_beta(S, R))
                crit = np.percentile(nulo, 95)
                fila = {"n": n_, "ruido": ruido, "c": cc, "metodo": met, "Δβ nulo, media": round(float(np.mean(nulo)), 3)}
                for tk in [0.5, 1.0]:
                    alt = []
                    for _ in range(R_):
                        idx = rng.choice(N, n_, replace=True)
                        S, R = resultado(met, *generar(rng, idx, ruido, cc, tk)); alt.append(delta_beta(S, R))
                    fila[f"poder techo {tk}"] = round(float(np.mean(np.array(alt) > crit)), 2)
                filas.append(fila)
    print("n =", n_, "listo;", round(time.time() - t_ini), "s")
tabla = pd.DataFrame(filas)
print(tabla.to_string(index=False))


## Paso C. Guardar

In [ ]:
tabla.to_csv("sim4_poder_por_n.csv", index=False)
with open("sim4_parametros.json", "w") as f:
    json.dump({"fecha": str(datetime.date.today()), "semilla": SEMILLA, "c_hat": C_HAT, "c_bajo": C_BAJO, "c_alto": C_ALTO,
               "d0_medio": float(prof.mean()), "pendiente_nadir_basal": float(b_n), "n_reps": N_REPS, "cohorte_n": int(N)}, f, indent=2)
print("Guardado: sim4_poder_por_n.csv y sim4_parametros.json")


## Cómo leer los resultados
- **c estimado:** si el intervalo incluye 0, no hay evidencia de acoplamiento y M2 es viable. Si es claramente mayor que 0, el método necesita más pacientes o un diseño distinto. Como la estimación es aproximada (tiende a exagerar), úsala como rango.
- **Tabla de poder:** se busca ≥ 0,80. Mira qué n lo alcanza para M2 con el c estimado. Eso orienta el tamaño que habría que pedir en INSPIRE.
- **Límites:** el modelo de caída y recuperación es una suposición; el ruido incluye cambios clínicos reales; es una simulación, no un resultado clínico.